In [ ]:
from pathlib import Path
from dotenv import load_dotenv
import os

from main import SAMPLE_FILES, build_metadata_for_path
from src import (
    ChunkingStrategyComparator,
    MarkdownStructureChunker,
    Document,
    EmbeddingStore,
    OpenAIEmbedder,
    LocalEmbedder,
    _mock_embed,
    compute_similarity,
)

load_dotenv(dotenv_path=Path('.env'), override=False)

def show_rows(rows, limit=None):
    rows = rows[:limit] if limit else rows
    try:
        import pandas as pd
        display(pd.DataFrame(rows))
    except Exception:
        for row in rows:
            print(row)

print('Ready')
print('Files:', len(SAMPLE_FILES))
print('EMBEDDING_PROVIDER=', os.getenv('EMBEDDING_PROVIDER', 'mock'))

## 1. Load Files

Kiểm tra danh sách file Markdown Viettel và số ký tự từng file.

In [ ]:
file_rows = []
for raw_path in SAMPLE_FILES:
    path = Path(raw_path)
    text = path.read_text(encoding='utf-8')
    file_rows.append({
        'file': path.name,
        'exists': path.exists(),
        'characters': len(text),
    })

show_rows(file_rows)

## 2. Build Metadata

`build_metadata_for_path()` trong `main.py` gán metadata document-level tối giản: `doc_id`, `domain`, `source`. `chunk_index` sẽ được thêm sau khi chunking.

In [ ]:
metadata_rows = []
for raw_path in SAMPLE_FILES:
    path = Path(raw_path)
    metadata = build_metadata_for_path(path)
    metadata_rows.append({
        'file': path.name,
        'doc_id': metadata.get('doc_id'),
        'domain': metadata.get('domain'),
        'source': metadata.get('source'),
    })

show_rows(metadata_rows)

## 3. Baseline Chunking Comparator

Chạy `ChunkingStrategyComparator().compare()` trên 3 tài liệu, đúng yêu cầu trong `exercises.md`.

In [ ]:
baseline_files = [
    'data/Viettel - MyViettel FAQs.md',
    'data/Viettel - Mobile FAQs.md',
    'data/Viettel - Internet - TV FAQs.md',
]

comparator = ChunkingStrategyComparator()
baseline_rows = []

for raw_path in baseline_files:
    path = Path(raw_path)
    text = path.read_text(encoding='utf-8')
    result = comparator.compare(text, chunk_size=1000)
    for strategy_name, stats in result.items():
        baseline_rows.append({
            'document': path.name,
            'strategy': strategy_name,
            'chunk_count': stats['count'],
            'avg_length': round(stats['avg_length'], 1),
        })

show_rows(baseline_rows)

## 4. Custom Markdown Structure Chunking

`MarkdownStructureChunker` tách theo heading Markdown, giữ Q/A section và thêm metadata cho từng chunk.

In [ ]:
chunker = MarkdownStructureChunker(chunk_size=1200)

sample_path = Path('data/Viettel - MyViettel FAQs.md')
sample_text = sample_path.read_text(encoding='utf-8')
sample_metadata = build_metadata_for_path(sample_path)
sample_records = chunker.chunk_with_metadata(sample_text, sample_metadata)

print('Sample file:', sample_path.name)
print('Chunks:', len(sample_records))
print('\nFirst chunk metadata:')
print(sample_records[0]['metadata'])
print('\nFirst chunk preview:')
print(sample_records[0]['content'][:500])

## 5. Convert Chunks To Documents

Mỗi chunk trở thành một `Document(id, content, metadata)` để đưa vào vector store.

In [ ]:
documents = []

for raw_path in SAMPLE_FILES:
    path = Path(raw_path)
    text = path.read_text(encoding='utf-8')
    base_metadata = build_metadata_for_path(path)
    records = chunker.chunk_with_metadata(text, base_metadata)
    for record in records:
        metadata = record['metadata']
        documents.append(Document(
            id=f"{metadata['doc_id']}_chunk_{metadata['chunk_index']}",
            content=record['content'],
            metadata=metadata,
        ))

print('Total chunk documents:', len(documents))
show_rows([
    {
        'id': doc.id,
        'domain': doc.metadata.get('domain'),
        'source': doc.metadata.get('source'),
        'chunk_index': doc.metadata.get('chunk_index'),
        'content_length': len(doc.content),
    }
    for doc in documents[:8]
])

## 6. Choose Embedding Backend

Mặc định lấy từ `.env`. Nếu OpenAI lỗi hoặc thiếu key, notebook fallback về `_mock_embed`.

In [ ]:
def create_embedder_from_env():
    provider = os.getenv('EMBEDDING_PROVIDER', 'mock').strip().lower()
    if provider == 'openai':
        try:
            return OpenAIEmbedder(model_name=os.getenv('OPENAI_EMBEDDING_MODEL', 'text-embedding-3-small'))
        except Exception as exc:
            print('OpenAI embedder fallback:', exc)
            return _mock_embed
    if provider == 'local':
        try:
            return LocalEmbedder(model_name=os.getenv('LOCAL_EMBEDDING_MODEL', 'all-MiniLM-L6-v2'))
        except Exception as exc:
            print('Local embedder fallback:', exc)
            return _mock_embed
    return _mock_embed

embedder = create_embedder_from_env()
print('Embedding backend:', getattr(embedder, '_backend_name', embedder.__class__.__name__))
test_vector = embedder('embedding smoke test')
print('Vector dimension:', len(test_vector))
print('First 5 values:', [round(v, 4) for v in test_vector[:5]])

## 7. Store Embeddings In Vector DB

`EmbeddingStore.add_documents()` embed từng chunk content và lưu vào ChromaDB nếu có, fallback in-memory nếu không.

In [ ]:
store = EmbeddingStore(collection_name='notebook_viettel_faq', embedding_fn=embedder)
store.add_documents(documents)

print('Use ChromaDB:', store._use_chroma)
print('Stored chunks:', store.get_collection_size())

## 8. Query Without Filter

Search toàn bộ vector store. Similarity được tính giữa query embedding và chunk content embeddings.

In [ ]:
query = 'Tôi không đăng nhập được app MyViettel thì phải làm gì?'
results = store.search(query, top_k=5)

show_rows([
    {
        'rank': idx,
        'score': round(result['score'], 4),
        'doc_id': result['metadata'].get('doc_id'),
        'domain': result['metadata'].get('domain'),
        'chunk_index': result['metadata'].get('chunk_index'),
    }
    for idx, result in enumerate(results, start=1)
])

## 9. Query With Metadata Filter

Filter không tham gia tính similarity. Nó chỉ giới hạn phạm vi search trước, ví dụ chỉ search trong `doc_id=viettel_myviettel_faqs`.

In [ ]:
filtered_results = store.search_with_filter(
    query,
    top_k=5,
    metadata_filter={'doc_id': 'viettel_myviettel_faqs'},
)

show_rows([
    {
        'rank': idx,
        'score': round(result['score'], 4),
        'doc_id': result['metadata'].get('doc_id'),
        'chunk_index': result['metadata'].get('chunk_index'),
        'preview': result['content'][:160].replace('\n', ' '),
    }
    for idx, result in enumerate(filtered_results, start=1)
])

## 10. Benchmark Queries

Chạy 5 benchmark queries với metadata filter hợp lý.

In [ ]:
benchmark_queries = [
    ('Tôi dùng BankPlus chuyển tiền nhầm thì có lấy lại tiền được không?', {'doc_id': 'viettel_digital_application_faqs'}),
    ('Camera Viettel có phân biệt chuyển động giữa người và vật không?', {'doc_id': 'viettel_digital_application_faqs'}),
    ('Hóa đơn điện tử là gì?', {'domain': 'business'}),
    ('Truyền hình cáp Viettel có chia được cho nhiều tivi không?', {'domain': 'internet'}),
    ('Tôi không đăng nhập được app MyViettel thì phải làm gì?', {'doc_id': 'viettel_myviettel_faqs'}),
]

benchmark_rows = []
for idx, (question, metadata_filter) in enumerate(benchmark_queries, start=1):
    top = store.search_with_filter(question, top_k=3, metadata_filter=metadata_filter)[0]
    benchmark_rows.append({
        '#': idx,
        'query': question,
        'filter': metadata_filter,
        'top1_score': round(top['score'], 4),
        'top1_doc_id': top['metadata'].get('doc_id'),
        'top1_chunk_index': top['metadata'].get('chunk_index'),
    })

show_rows(benchmark_rows)

## 11. Similarity Predictions

Tính cosine similarity trên embeddings của 5 cặp câu.

In [ ]:
pairs = [
    ('MyViettel kiểm tra gói data', 'Ứng dụng My Viettel hiển thị gói Mobile Internet và lưu lượng còn lại', 'high'),
    ('Đăng ký gói 4G', 'Soạn tin nhắn tên gói cước gửi 191 để đăng ký', 'high'),
    ('Hóa đơn điện tử doanh nghiệp', 'Doanh nghiệp có thể tra cứu và sử dụng hóa đơn điện tử', 'high'),
    ('Lắp internet gia đình', 'Dịch vụ Internet và truyền hình Viettel hỗ trợ khách hàng gia đình', 'high'),
    ('Cách đăng nhập MyViettel', 'Cửa hàng bán điện thoại iPhone và phụ kiện', 'low'),
]

similarity_rows = []
for idx, (a, b, prediction) in enumerate(pairs, start=1):
    score = compute_similarity(embedder(a), embedder(b))
    similarity_rows.append({
        '#': idx,
        'sentence_a': a,
        'sentence_b': b,
        'prediction': prediction,
        'actual_score': round(score, 4),
    })

show_rows(similarity_rows)

## 12. RAG Prompt Injection

Cell này lấy retrieved chunks, build prompt có context, rồi gọi OpenAI Chat nếu có `OPENAI_API_KEY`. Nếu không muốn gọi LLM, chỉ xem prompt là đủ.

In [ ]:
rag_question = 'Tôi không đăng nhập được app MyViettel thì phải làm gì?'
rag_context = store.search_with_filter(
    rag_question,
    top_k=3,
    metadata_filter={'doc_id': 'viettel_myviettel_faqs'},
)

context_blocks = []
for idx, result in enumerate(rag_context, start=1):
    md = result['metadata']
    context_blocks.append(
        f"[{idx}] Source: {md.get('source')}\n"
        f"Chunk index: {md.get('chunk_index')}\n"
        f"Score: {result['score']:.4f}\n"
        f"{result['content']}"
    )

prompt = (
    'Hãy trả lời câu hỏi bằng tiếng Việt, ngắn gọn và chỉ dựa trên context dưới đây.\n'
    'Nếu context không đủ, hãy nói rõ là chưa đủ thông tin.\n\n'
    f"Context:\n{'\n\n'.join(context_blocks)}\n\n"
    f"Câu hỏi: {rag_question}\n"
    'Trả lời:'
)

print(prompt[:2500])

In [ ]:
RUN_LLM = False  # đổi thành True nếu muốn gọi OpenAI Chat API

if RUN_LLM:
    from openai import OpenAI
    client = OpenAI()
    response = client.chat.completions.create(
        model=os.getenv('OPENAI_CHAT_MODEL', 'gpt-4o-mini'),
        messages=[
            {'role': 'system', 'content': 'Bạn là trợ lý CSKH Viettel. Trả lời dựa trên context.'},
            {'role': 'user', 'content': prompt},
        ],
        temperature=0.2,
    )
    print(response.choices[0].message.content)
else:
    print('RUN_LLM=False, chưa gọi LLM. Prompt ở cell trên là phần được inject context.')